# W03 Exercise: Baselines, target types, and model assumptions

Machine Learning in Engineering Science | NCKU / LAiMM | Fall 2026

**Date:** 2026-09-21. **Exercise time:** 35 minutes within the class/practice allocation, not extra time after class.

**Classroom formative practice. No new weekly graded submission is created.**

## How to work independently
1. In Colab, save a copy to your Drive and use a CPU runtime. In Jupyter, open this notebook and select a Python kernel.
2. Run the complete worked walkthrough below from top to bottom. Read every Markdown checkpoint; the walkthrough is included so no previous notebook state is needed.
3. At the Independent Exercise section, write each prediction in the response cell before running its experiment.
4. Run the starter experiment, make the requested controlled change, and record the actual result. Edit only the declared factor; preserve split rules and preprocessing.
5. Explain one mechanism and one limitation. Export the evidence JSON and discuss it with a partner.

The core exercises need no paid API, private dataset, or GPU. The agent units use clearly labeled offline simulators, not real LLM calls.

**Reproducibility boundary:** the original guided notebooks were previously executed; the new extension cells have not yet been execution-tested in this revision. Report unexpected behavior rather than claiming a check passed.


# Week 03: Regression, classification, and baselines

Machine Learning in Engineering Science | NCKU + LAiMM | Fall 2026

**Purpose:** a fully worked, self-paced in-class practice with an optional extension. Weeks 11 and 18 are optional examination-readiness materials. These notebooks do not create weekly graded assignments.

**Before running:** choose a Python 3 kernel in Jupyter or a CPU runtime in Colab. Run every cell from top to bottom with Shift+Enter. Setup installs missing free packages. No API key or external dataset download is required.

**Learning target:** complete the calculation, inspect intermediate values, and explain why the protocol supports a limited claim.

**Time:** about 25-45 minutes including reading, prediction, execution, and explain-back.


## 1. Set up and record the environment

Keep the seed fixed for the first run. Record actual outputs and package versions rather than copying a claimed performance number.


In [ ]:
import sys, subprocess, importlib.util, platform, json, random, hashlib
from pathlib import Path
required = {"numpy": "numpy", "matplotlib": "matplotlib", "sklearn": "scikit-learn"}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
import numpy as np
import matplotlib.pyplot as plt
SEED = 2026
random.seed(SEED)
np.random.seed(SEED)
rng = np.random.default_rng(SEED)
ARTIFACTS = Path("artifacts")
ARTIFACTS.mkdir(exist_ok=True)
print("Python:", platform.python_version(), "NumPy:", np.__version__, "Seed:", SEED)


**Checkpoint.** The environment cell must finish before any analysis cell. Restart and run all if a variable is missing.


## 2. Generate a regression problem

The target is continuous. Generate data from a known linear mechanism plus noise, then hold out test rows before fitting.


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge,LogisticRegression
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error,accuracy_score
X=rng.normal(size=(240,3)); y=3*X[:,0]-2*X[:,1]+rng.normal(0,.5,240)
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.25,random_state=SEED)
baseline=DummyRegressor().fit(Xtr,ytr); regression=Ridge(alpha=1).fit(Xtr,ytr)
print("Mean baseline MAE:",mean_absolute_error(yte,baseline.predict(Xte)))
print("Ridge MAE:",mean_absolute_error(yte,regression.predict(Xte)))


**Checkpoint.** MAE is in target units. Predicting the training mean is a meaningful baseline, even when the fitted model performs better.


## 3. Change the question to classification

Define whether the response exceeds zero. This changes the prediction target and metric; it does not turn a regression output into a calibrated probability.


In [ ]:
labels=(y>0).astype(int)
Atr,Ate,btr,bte=train_test_split(X,labels,test_size=.25,stratify=labels,random_state=SEED)
classifier=LogisticRegression(random_state=SEED).fit(Atr,btr)
probability=classifier.predict_proba(Ate)[:,1]
summary={"ridge_test_mae":float(mean_absolute_error(yte,regression.predict(Xte))),
         "classification_accuracy":float(accuracy_score(bte,probability>=.5))}
print(summary)


**Checkpoint.** Describe the event represented by class 1 and why accuracy and MAE answer different questions.


## 4. Plot predicted versus observed

A diagonal reference line shows perfect predictions. Residual scatter conveys error structure.


In [ ]:
prediction=regression.predict(Xte)
plt.scatter(yte,prediction); limits=[min(yte),max(yte)]
plt.plot(limits,limits,"k--"); plt.xlabel("Observed synthetic response")
plt.ylabel("Predicted response"); plt.show()
print("Ridge coefficients:",regression.coef_)


**Checkpoint.** Compare learned coefficients with the generating coefficients [3,-2,0]. Differences reflect sample size, noise, and regularization.


## 6. Preserve your evidence

Export a JSON record with a seed, package version, and measured result. Download the artifacts directory before leaving Colab; runtime storage is temporary.


In [ ]:
record={"week":3,"seed":SEED,"numpy":np.__version__,"python":platform.python_version(),"result":summary}
(ARTIFACTS/"week-03-evidence.json").write_text(json.dumps(record,indent=2))
print(json.dumps(record,indent=2))


**Checkpoint.** The evidence file must match the result you actually obtained. A saved screenshot alone does not permit rerunning the calculation.


## 7. Try one change and explain it
1. Predict the effect of one change before editing code. Change one parameter, input, or boundary condition only.
2. Rerun all cells in a fresh kernel. Keep the original evidence record and compare the observed difference.
3. Explain the result using the mechanism, not just the final number.
4. Name one assumption that limits the claim, and a new test that could challenge it.

**Troubleshooting:** if installation fails, restore network access; if a variable is undefined, run earlier cells; if your output differs, compare package versions, seed, and data construction. Do not repeatedly edit code until a preferred score appears.

**Completion criterion:** one executed notebook, one evidence JSON, and a brief claim/evidence/limitation statement.

### Reference documentation
- [NumPy user guide](https://numpy.org/doc/stable/user/)
- [scikit-learn user guide](https://scikit-learn.org/stable/user_guide.html)
- [Course repository](https://github.com/CHjonnyYu/MLES-Fall-2026)


## Independent Exercise: W03

Complete the three tasks below after the walkthrough. The starter cells produce observations, not your written interpretation. Fill the response fields in the next cell. Save separate records for the original run and any controlled change.

In [ ]:
exercise_id="W03"
observations={}
responses={
    'task1_prediction':'', 'task1_explanation':'',
    'task2_prediction':'', 'task2_explanation':'',
    'task3_prediction':'', 'task3_explanation':'',
    'controlled_change':'', 'limitation':'', 'ai_use_and_checks':''
}
# Fill these strings before and after each experiment. Use 'none' if no AI assistance was used.
print('Exercise record initialized:', exercise_id)

### Task 1: Beat a declared regression baseline

Using the worked notebook, compare the fitted regression with a training-mean predictor on the same test rows. Do not fit or select the model using test outcomes.

**Steps**
1. Write your prediction in `responses['task1_prediction']` before running.
2. Run the starter cell; read every printed quantity and retain the observation.
3. Make the controlled comparison requested above. Record its configuration in `responses['controlled_change']`.
4. Write a mechanism-based explanation in `responses['task1_explanation']`; do not merely repeat a score.

**Checkpoint:** Both predictions must correspond to the same yte. Lower MAE is better for this declared metric, not proof of universal superiority.

In [ ]:
from sklearn.metrics import mean_absolute_error
reg_pred=regression.predict(Xte)
mean_pred=np.full(len(yte),float(np.mean(ytr)))
observations['task1']={'ridge_MAE':float(mean_absolute_error(yte,reg_pred)),'mean_baseline_MAE':float(mean_absolute_error(yte,mean_pred)),'test_rows':len(yte)}
print(observations['task1'])

### Task 2: Change the target question

Compare the number of positive labels under thresholds 0 and 2. Explain why this changes the learning task rather than merely changing a display option.

**Steps**
1. Write your prediction in `responses['task2_prediction']` before running.
2. Run the starter cell; read every printed quantity and retain the observation.
3. Make the controlled comparison requested above. Record its configuration in `responses['controlled_change']`.
4. Write a mechanism-based explanation in `responses['task2_explanation']`; do not merely repeat a score.

**Checkpoint:** There are three positives at threshold 0 and one at threshold 2. A continuous target and a binary event require different metrics.

In [ ]:
target_demo=np.array([-2.,-1.,0.,1.,2.,3.])
observations['task2']={'threshold_0':(target_demo>0).astype(int).tolist(),'threshold_2':(target_demo>2).astype(int).tolist()}
print(observations['task2'])

### Task 3: Interpret coefficients with limits

Compare fitted coefficients with the synthetic generating coefficients. Explain the third feature and why a nonzero fitted coefficient is not automatically causal evidence.

**Steps**
1. Write your prediction in `responses['task3_prediction']` before running.
2. Run the starter cell; read every printed quantity and retain the observation.
3. Make the controlled comparison requested above. Record its configuration in `responses['controlled_change']`.
4. Write a mechanism-based explanation in `responses['task3_explanation']`; do not merely repeat a score.

**Checkpoint:** Noise, finite data, correlations, and regularization can change fitted coefficients. Record an observed discrepancy rather than inventing an exact match.

In [ ]:
observations['task3']={'fitted_coefficients':regression.coef_.tolist(),'generating_coefficients':[3,-2,0]}
print(observations['task3'])

### Finish: preserve and explain

Fill all response strings, then run the export cell. A draft file is intentionally labeled incomplete; a complete response record is not automatically a scientifically correct result. Download the JSON before the Colab session ends. Share it with a partner for explain-back, not as an additional weekly submission.

**Exit questions:** What changed? Which observation supports your explanation? What does this experiment not establish?

In [ ]:
missing_responses=[key for key,value in responses.items() if not str(value).strip()]
exercise_record={'exercise':exercise_id,'status':'draft' if missing_responses else 'ready for peer discussion','responses':responses,'observations':observations,'missing_responses':missing_responses,'environment':{'python':platform.python_version(),'numpy':np.__version__},'data_scope':'course-provided or synthetic teaching data'}
exercise_folder=Path('artifacts')/'exercises'; exercise_folder.mkdir(parents=True,exist_ok=True)
exercise_path=exercise_folder/(exercise_id+'-evidence.json')
exercise_path.write_text(json.dumps(exercise_record,indent=2,default=lambda value:value.tolist() if hasattr(value,'tolist') else str(value)))
print('Saved:',exercise_path)
print('Status:',exercise_record['status'])
print('Fields still to complete:',missing_responses)